# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management, but install **into Colab's system Python** (via `uv pip install --system`) so the running kernel picks up the packages directly — no venv, no kernel switch.

The cell below:
1. Installs `uv` itself with `pip` (one-shot, no PATH juggling).
2. Installs `vllm`, `bitsandbytes`, `transformers`, etc. in parallel.
3. Is **idempotent** — re-running it when `vllm` is already present is a no-op, so you can re-run any cell safely.

> **If `import vllm` fails in Section 2**, use *Runtime → Restart runtime* once, then continue from Section 2. You do **not** need to re-run this cell after the restart (packages are already on disk).

## 1.5 Colab Pro+ Background Execution & Data Persistence

**Before running the heavy cells below**, complete these three steps:

### 1) Enable Background Execution
You have Colab Pro+ — sessions continue running even after you close the browser tab.
- Click the **gear icon** (top-right) → **Miscellaneous** → make sure **"Automatically close idle sessions"** is **unchecked**.
- As long as a cell is actively running, the VM stays alive in the background.

### 2) Runtime Limits
| Limit | Pro+ Typical |
|---|---|
| Max session duration | ~24 hours |
| Compute-unit burn rate (RTX Pro 6000) | ~8.71 units / hour |
| Your remaining units | Check top-right badge |

Plan your run so it finishes well within 24 h. If you run out of compute units mid-run, the VM terminates **without warning**.

### 3) Data Persistence — Google Drive Auto-Save
Colab's `/content` disk is **ephemeral** — everything vanishes when the session ends.  
The cell below mounts Google Drive and creates a project directory so that **all outputs (results, checkpoints, model artifacts) are written directly to Drive** and survive session termination.

*Supplementary section: prose drafted with an AI coding assistant (not part of the original course starter).*

In [ ]:
# Supplementary Colab helper — written with an AI coding assistant (not original starter code).
import importlib.util
from pathlib import Path

DRIVE_PROJECT_ROOT: str | None = None

if importlib.util.find_spec("google.colab") is not None:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/151b_smallLM"
    drive_root = Path(DRIVE_PROJECT_ROOT)
    (drive_root / "results").mkdir(parents=True, exist_ok=True)
    (drive_root / "checkpoints").mkdir(parents=True, exist_ok=True)
    (drive_root / "models").mkdir(parents=True, exist_ok=True)

    import os

    if drive_root.is_dir():
        os.chdir(drive_root)
        print(f"Working directory set to project root (so judger.py resolves): {drive_root}")

    print(f"Google Drive mounted. Project root: {DRIVE_PROJECT_ROOT}")
    print(f"  results/      → {drive_root / 'results'}")
    print(f"  checkpoints/  → {drive_root / 'checkpoints'}")
    print(f"  models/       → {drive_root / 'models'}")
    print(
        "\nTip: The notebook writes MCQ/free-form JSONL next to OUTPUT_PATH (usually "
        "results/mcq_gen_checkpoint.jsonl, …), not inside checkpoints/ (that folder is for model weights)."
    )

    import shutil
    total, used, free = shutil.disk_usage(str(drive_root))
    print(f"\nDrive space: {free / (1024**3):.1f} GB free / {total / (1024**3):.1f} GB total")
else:
    print("Not running on Colab — skipping Drive mount. Outputs will be saved locally.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Dependency install (Colab-optimized, idempotent)
#
# Design choices:
#   • Use `uv pip install --system` so packages land in the SAME Python
#     that Colab's kernel already uses (no venv/kernel switch required).
#   • Pip-install uv itself first (the curl installer puts uv in ~/.local/bin
#     which is not on PATH in the running shell — common failure mode).
#   • Pin bitsandbytes + antlr4 (vLLM + sympy are picky about these).
#   • Skip reinstall when vllm is already importable — avoids re-downloading
#     ~5 GB of wheels when a cell is rerun.
# ─────────────────────────────────────────────────────────────────────────────
import importlib.util, subprocess, sys

def _sh(cmd: str) -> None:
    """Run a shell command and stream output live."""
    print(f"$ {cmd}")
    subprocess.check_call(cmd, shell=True)

if importlib.util.find_spec("vllm") is None:
    _sh(f"{sys.executable} -m pip install -q --upgrade pip uv")
    _sh(
        f"{sys.executable} -m uv pip install --system -q "
        "sympy numpy tqdm "
        "'transformers>=4.45' "
        "'vllm>=0.6.3' "
        "'bitsandbytes>=0.44' "
        "'antlr4-python3-runtime==4.11.1' "
        "ipykernel"
    )
    print("\nInstall complete. If `import vllm` still fails in the next cell, "
          "use Runtime → Restart runtime, then re-run from here.")
else:
    print("vllm already installed — skipping reinstall.")

# judger.py imports sympy.parse_latex, which requires antlr4 4.11.x — not always present when vLLM was pre-installed.
_sh(f"{sys.executable} -m pip install -q 'antlr4-python3-runtime==4.11.1'")

# Quick sanity print (these imports will actually succeed in the NEXT cell
# after a kernel restart, if a restart was needed).
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv 2>/dev/null || echo "nvidia-smi not available (no GPU in this runtime)"

## 2. Imports & Configuration

**Google Colab:** Run the next cell once. If it installs packages, use **Runtime → Restart runtime** and run that cell again if `vllm` still fails to import.

**Data on Google Drive:** Colab’s default folder is `/content`, not your Drive. The config cell **mounts Drive** (authorize when prompted). For **public** it looks for `MyDrive/151b_smallLM/data/public.jsonl`. For **Kaggle private** inference, place `cse-151-b-spring-2026-competition/private.jsonl` under `MyDrive/151b_smallLM/` (same layout as your local repo) and set `DATA_SPLIT = "private"` below.

All key settings are collected in one place.  
- `DATA_SPLIT` — `"public"` (default) for `public.jsonl` with answers; `"private"` for Kaggle `private.jsonl` (no answers). Sets `SAVE_EVAL` automatically and changes `OUTPUT_PATH` to `results/private_inference.jsonl`.
- `DATA_PATH` — resolved automatically from `DATA_SPLIT` (local `data/` or Drive paths above)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response
- `SUBSET_SIZE` — `None` = all questions; or a small integer for a quick smoke test
- `GREEDY` — `True` → `temperature=0` (deterministic single sample)
- `SEED` — optional RNG seed for `SamplingParams` (`None` to omit)
- `MCQ_SELF_CONSISTENCY_N` — `1` = one sample; `>1` = multiple samples for **MCQ only**, then majority vote on the letter inside `\boxed{}`
- `MCQ_CHECKPOINT_PATH`, `SKIP_MCQ_GENERATION` — after a full MCQ run, responses are saved to the checkpoint file. Set `SKIP_MCQ_GENERATION=True` on a **later GPU session** to reload MCQ and run **only** free-form (saves roughly the MCQ wall time).
- `RELOAD_RESPONSES_FROM_SNAPSHOT` — set `True` to skip **all** vLLM generation and load `responses_snapshot.jsonl` next to `OUTPUT_PATH` (same `DATA_PATH` / `SUBSET_SIZE` as the run that wrote the snapshot). Use this to score or export after an overnight run without burning GPU again.
- `GPU_MEMORY_UTILIZATION`, `MAX_MODEL_LEN`, `MAX_NUM_SEQS`, `MAX_NUM_BATCHED_TOKENS` — vLLM memory / batching (lower if you OOM)

In [ ]:
import importlib

import importlib.util

import json

import os

import re

import subprocess

import sys

from collections import Counter

from pathlib import Path

from typing import Any, Optional





def _ensure_competition_packages() -> None:

    """Install deps when missing. Mirrors Section 1; safe to run standalone on a fresh Colab runtime."""

    try:

        importlib.import_module("vllm")

    except ModuleNotFoundError:

        subprocess.check_call(

            [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "uv"]

        )

        subprocess.check_call(

            [

                sys.executable, "-m", "uv", "pip", "install", "--system", "-q",

                "sympy", "numpy", "tqdm",

                "transformers>=4.45",

                "vllm>=0.6.3",

                "bitsandbytes>=0.44",

                "antlr4-python3-runtime==4.11.1",

            ]

        )

        importlib.invalidate_caches()

        try:

            importlib.import_module("vllm")

        except ModuleNotFoundError as exc:

            raise RuntimeError(

                "vllm installed but not importable yet. "

                "Use Runtime → Restart runtime, then re-run this cell."

            ) from exc



    # judger.py → sympy.parse_latex needs this even when vLLM was already on the image.

    subprocess.check_call(

        [

            sys.executable,

            "-m",

            "pip",

            "install",

            "-q",

            "antlr4-python3-runtime==4.11.1",

        ]

    )





_ensure_competition_packages()



from transformers import AutoTokenizer

from vllm import LLM, SamplingParams

from tqdm import tqdm





# "public" = answers available for local scoring. # "private" = Kaggle ``private.jsonl`` (no answers) for leaderboard submission.

DATA_SPLIT = "public"  # set to "private" on Colab when running Kaggle test inference





def _resolve_data_and_output_paths() -> tuple[str, str]:

    """Resolve ``DATA_PATH`` / ``OUTPUT_PATH`` for public (with answers) or Kaggle private (no answers)."""

    split = globals().get("DATA_SPLIT", "public")

    if split not in ("public", "private"):

        raise ValueError('DATA_SPLIT must be "public" or "private"')



    if split == "private":

        local_priv = [

            Path("cse-151-b-spring-2026-competition/private.jsonl"),

            Path("data/private.jsonl"),

        ]

        for p in local_priv:

            if p.is_file():

                return str(p), str(Path("results/private_inference.jsonl"))



        if importlib.util.find_spec("google.colab") is None:

            raise FileNotFoundError(

                "DATA_SPLIT is private but private.jsonl not found. Expected one of:\n  "

                + "\n  ".join(str(x) for x in local_priv)

                + "\nCopy Kaggle ``private.jsonl`` into the repo (see course layout) or set DATA_SPLIT to public."

            )



        from google.colab import drive



        drive.mount("/content/drive", force_remount=False)

        drive_priv = [

            Path("/content/drive/MyDrive/151b_smallLM/cse-151-b-spring-2026-competition/private.jsonl"),

            Path("/content/drive/MyDrive/151b_smallLM/data/private.jsonl"),

            Path("/content/drive/MyDrive/151B_smallLM/cse-151-b-spring-2026-competition/private.jsonl"),

        ]

        for candidate in drive_priv:

            if candidate.is_file():

                project_root = candidate.parent.parent

                out = project_root / "results" / "private_inference.jsonl"

                return str(candidate), str(out)



        raise FileNotFoundError(

            "DATA_SPLIT is private but private.jsonl not found on Drive. Expected one of:\n  "

            + "\n  ".join(str(p) for p in drive_priv)

            + "\nPlace ``cse-151-b-spring-2026-competition/private.jsonl`` under MyDrive/151b_smallLM/."

        )



    # --- public split (original behaviour) ---

    local_data = Path("data/public.jsonl")

    if local_data.is_file():

        return str(local_data), "results/starter_results.jsonl"



    if importlib.util.find_spec("google.colab") is None:

        return str(local_data), "results/starter_results.jsonl"



    from google.colab import drive



    drive.mount("/content/drive", force_remount=False)

    drive_candidates = [

        Path("/content/drive/MyDrive/151b_smallLM/data/public.jsonl"),

        Path("/content/drive/MyDrive/151B_smallLM/data/public.jsonl"),

    ]

    for candidate in drive_candidates:

        if candidate.is_file():

            project_root = candidate.parent.parent

            out = project_root / "results" / "starter_results.jsonl"

            return str(candidate), str(out)



    raise FileNotFoundError(

        "Could not find public.jsonl on Google Drive. Expected one of:\n  "

        + "\n  ".join(str(p) for p in drive_candidates)

        + "\nPlace the repo as MyDrive/151b_smallLM/ with data/public.jsonl inside, "

        "or add your path to drive_candidates in this cell."

    )





# ── Configuration ─────────────────────────────────────────────────────────────

MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"

# Path to a LoRA adapter directory (containing adapter_config.json + adapter_model.safetensors).
# Examples:
#   ADAPTER_PATH = "checkpoints/sft/final"
#   ADAPTER_PATH = "/content/drive/MyDrive/151b_smallLM/checkpoints/grpo/final"
# Set to None to run the base MODEL_ID without any adapter.
ADAPTER_PATH: Optional[str] = None
# Must be >= the LoRA rank used during training (configs/sft_config.yaml uses r=64).
MAX_LORA_RANK = 64

GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES

DATA_PATH, OUTPUT_PATH = _resolve_data_and_output_paths()

# Auto: private split has no labels — Section 7 skips Judger; Section 9 omits gold/correct.

SAVE_EVAL = DATA_SPLIT == "public"

if not Path(DATA_PATH).is_file():

    raise FileNotFoundError(

        f"Dataset not found: {DATA_PATH}\n"

        "Clone the repo locally so data/public.jsonl exists, or on Colab use Drive paths above."

    )

print(f"Using DATA_PATH={DATA_PATH}")

print(f"Using OUTPUT_PATH={OUTPUT_PATH}")

MAX_TOKENS  = 32768

# Smoke test: set to a small int (e.g. 5). Full public eval: None

SUBSET_SIZE: int | None = None



GREEDY = False                     # True: temperature 0, single-sample decoding

SEED: Optional[int] = 42           # None: let vLLM default

MCQ_SELF_CONSISTENCY_N = 8         # >1: extra MCQ samples + majority vote

FREEFORM_SELF_CONSISTENCY_N = 8    # >1: extra free-form samples + majority vote on boxed answer



GPU_MEMORY_UTILIZATION = 0.90

MAX_MODEL_LEN = 32768

MAX_NUM_SEQS = 256

MAX_NUM_BATCHED_TOKENS = 65536



# Two-session workflow: MCQ run writes this file; next session set SKIP_MCQ_GENERATION=True.

MCQ_CHECKPOINT_PATH = str(Path(OUTPUT_PATH).with_name("mcq_gen_checkpoint.jsonl"))

SKIP_MCQ_GENERATION = False  # True only if MCQ_CHECKPOINT_PATH exists from a previous run



# When True, skips all vLLM generation and loads answers from ``responses_snapshot.jsonl``

# next to OUTPUT_PATH (same DATA_PATH / SUBSET_SIZE as the overnight run). Use for scoring only.

RELOAD_RESPONSES_FROM_SNAPSHOT = False



# Do not set CUDA_VISIBLE_DEVICES to "" — that hides all GPUs and vLLM raises

# RuntimeError: Device string must not be empty. Use "0" (or "1", ...) on CUDA hosts.

_gpu = str(GPU_ID).strip()

if _gpu:

    os.environ["CUDA_VISIBLE_DEVICES"] = _gpu

else:

    os.environ.pop("CUDA_VISIBLE_DEVICES", None)


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician who solves problems with rigorous step-by-step reasoning.\n\n"
    "Instructions:\n"
    "1. Break the problem into clear steps. Show all work.\n"
    "2. For each step, state what you are doing and why.\n"
    "3. Double-check your arithmetic and algebra before concluding.\n"
    "4. Put your final answer inside \\boxed{}.\n"
    "5. If the problem asks for multiple sub-answers, separate them by commas "
    "inside a single \\boxed{}, e.g. \\boxed{3, 7}.\n"
    "6. Give exact answers (fractions, radicals, pi) unless a decimal approximation is explicitly requested.\n"
    "7. If the answer is a well-known constant or expression, simplify it fully."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician who solves problems with rigorous step-by-step reasoning.\n\n"
    "Instructions:\n"
    "1. First, solve the problem yourself step-by-step WITHOUT looking at the options.\n"
    "2. Then compare your solution to each option to find the match.\n"
    "3. If your answer does not match any option, re-examine your work and "
    "try alternative approaches.\n"
    "4. Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        user_prompt = (
            f"{question}\n\n"
            f"Options:\n{opts_text}\n\n"
            f"Solve the problem step-by-step, then select the correct option letter."
        )
        return SYSTEM_PROMPT_MCQ, user_prompt
    user_prompt = (
        f"{question}\n\n"
        f"Solve this problem step-by-step, showing all work. "
        f"Put your final answer in \\boxed{{}}."
    )
    return SYSTEM_PROMPT_MATH, user_prompt


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 300 chars) ──")
    print(usr_p[:300], "...\n")

## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **4-bit NF4 weight-only quantization** via BitsAndBytes (`quantization="bitsandbytes"` + `load_format="bitsandbytes"` → vLLM's default bnb backend, which is 4-bit, *not* INT8). This cuts weight memory by ~4× vs BF16, leaving more room for the KV cache.

`enable_prefix_caching=True` is important here: the MCQ pass reuses the same ~200-token system prompt across all questions, and prefix caching turns that into a free speedup.

Key parameters (tune in the config cell): `GPU_MEMORY_UTILIZATION`, `MAX_MODEL_LEN`, `MAX_NUM_SEQS`, `MAX_NUM_BATCHED_TOKENS`.

In [ ]:
import torch

if RELOAD_RESPONSES_FROM_SNAPSHOT:
    # Scoring-only path: skip multi-GB vLLM weights; generation cell loads answers from disk.
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    llm = None  # type: ignore[assignment]
    print("RELOAD_RESPONSES_FROM_SNAPSHOT: skipping vLLM LLM(...) — tokenizer only.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "PyTorch does not see any CUDA GPU (torch.cuda.is_available() is False). "
            "vLLM then gets an empty device type → RuntimeError: Device string must not be empty.\n\n"
            "What to do:\n"
            "  • Google Colab: Runtime → Change runtime type → Hardware accelerator → GPU → Save, "
            "reconnect, rerun from the config cell.\n"
            "  • Run !nvidia-smi in a cell; if it fails, there is no GPU in this session.\n"
            "  • Rerun the config cell so CUDA_VISIBLE_DEVICES is set (never leave GPU_ID as '')."
        )

    print(f"CUDA OK: {torch.cuda.device_count()} device(s), {torch.cuda.get_device_name(0)}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token

    llm_kwargs: dict = dict(
        model=MODEL_ID,
        quantization="bitsandbytes",
        load_format="bitsandbytes",
        # MCQ pass reuses the same system prompt for every question -> prefix cache
        # gives a large, free speedup. (Previously set to False as a workaround for
        # older vllm+bnb issues; fine on vllm>=0.6.3.)
        enable_prefix_caching=True,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        max_model_len=MAX_MODEL_LEN,
        trust_remote_code=True,
        max_num_seqs=MAX_NUM_SEQS,
        max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS,
    )
    # Enable LoRA only when an adapter directory is configured, so unmodified runs
    # behave exactly as before.
    if ADAPTER_PATH:
        llm_kwargs["enable_lora"] = True
        llm_kwargs["max_lora_rank"] = MAX_LORA_RANK
        print(f"LoRA enabled: adapter={ADAPTER_PATH}, max_lora_rank={MAX_LORA_RANK}")

    llm = LLM(**llm_kwargs)

    print("Model loaded. Sampling params are built in the next section.")

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()`.

- **Default** — one forward pass over all prompts with `n=1`.
- **`MCQ_SELF_CONSISTENCY_N > 1`** — MCQ prompts use `n>1` (stochastic samples); we keep **one** response string per question by majority vote on the extracted `\boxed{}` letter. Mixed MCQ + free-form runs **two** passes (MCQ with `n>1`, then free-form with `FREEFORM_SELF_CONSISTENCY_N`).
- **`GREEDY`** — `temperature=0`. If you also set `MCQ_SELF_CONSISTENCY_N > 1`, the MCQ pass temporarily uses stochastic decoding so samples are not identical.

**GPU time limit:** After MCQ finishes, a checkpoint is written to `MCQ_CHECKPOINT_PATH`. If you disconnect or start a new rental, set `SKIP_MCQ_GENERATION=True` in the config cell (and keep the same `data` / `SUBSET_SIZE`) so only the free-form pass uses the GPU.

In [ ]:
def extract_letter(text: str) -> str:
    """Extract a single letter answer from \\boxed{} or fallback to last capital letter."""
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def extract_boxed_answer(text: str) -> str:
    """Extract the raw content inside the last \\boxed{} in text."""
    # Strip thinking tags
    think_end = text.rfind("</think>")
    search_text = text[think_end + len("</think>"):] if think_end >= 0 else text

    idx = search_text.rfind("\\boxed{")
    if idx < 0:
        return ""
    brace_start = idx + len("\\boxed{")
    depth = 1
    i = brace_start
    while i < len(search_text) and depth > 0:
        if search_text[i] == '{':
            depth += 1
        elif search_text[i] == '}':
            depth -= 1
        i += 1
    if depth == 0:
        return search_text[brace_start:i - 1].strip()
    return ""


def pick_mcq_majority_text(completion_outputs: list[Any]) -> str:
    """Return one completion text: majority vote on boxed letter; tie -> first winning sample."""
    outs = list(completion_outputs)
    letters = [extract_letter(o.text.strip()) for o in outs]
    valid = [L for L in letters if L]
    if not valid:
        return outs[0].text.strip()
    winner, _ = Counter(valid).most_common(1)[0]
    for o, L in zip(outs, letters):
        if L == winner:
            return o.text.strip()
    return outs[0].text.strip()


def pick_freeform_majority_text(completion_outputs: list[Any]) -> str:
    """Return one completion text: majority vote on boxed answer string for free-form."""
    outs = list(completion_outputs)
    answers = [extract_boxed_answer(o.text.strip()) for o in outs]
    valid = [(ans, i) for i, ans in enumerate(answers) if ans]
    if not valid:
        return outs[0].text.strip()

    # Normalize whitespace for comparison
    normalized = [re.sub(r"\s+", " ", a).strip().lower() for a, _ in valid]
    counter = Counter(normalized)
    winner_norm, _ = counter.most_common(1)[0]

    for (ans, idx), norm in zip(valid, normalized):
        if norm == winner_norm:
            return outs[idx].text.strip()
    return outs[0].text.strip()


def make_sampling_params(*, n: int, force_stochastic: bool) -> SamplingParams:
    """Build vLLM sampling params. When ``force_stochastic``, ignore ``GREEDY`` (for n>1)."""
    if GREEDY and not force_stochastic:
        temperature = 0.0
        top_p = 1.0
        top_k = -1
    else:
        temperature = 0.7
        top_p = 0.95
        top_k = 40
    kwargs: dict[str, Any] = {
        "max_tokens": MAX_TOKENS,
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
        "min_p": 0.0,
        "presence_penalty": 0.0,
        "repetition_penalty": 1.0,
        "n": n,
    }
    if SEED is not None:
        kwargs["seed"] = SEED
    return SamplingParams(**kwargs)


work_data = data[:SUBSET_SIZE] if SUBSET_SIZE is not None else data
mcq_idx = [i for i, item in enumerate(work_data) if item.get("options")]
free_idx = [i for i, item in enumerate(work_data) if not item.get("options")]

if MCQ_SELF_CONSISTENCY_N < 1:
    raise ValueError("MCQ_SELF_CONSISTENCY_N must be >= 1")
if FREEFORM_SELF_CONSISTENCY_N < 1:
    raise ValueError("FREEFORM_SELF_CONSISTENCY_N must be >= 1")

mcq_n = MCQ_SELF_CONSISTENCY_N
free_n = FREEFORM_SELF_CONSISTENCY_N


def prompt_for_item(item: dict) -> str:
    """Format a single item into a full chat-template prompt string."""
    system, user = build_prompt(item["question"], item.get("options"))
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )


def _load_mcq_checkpoint(path: str) -> dict[str, str]:
    """Load ``id -> response`` from JSONL written after the MCQ pass."""
    result: dict[str, str] = {}
    with open(path, encoding="utf-8") as fp:
        for line in fp:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            result[str(row["id"])] = row["response"]
    return result


responses = [""] * len(work_data)

FREEFORM_CHECKPOINT_PATH = str(Path(OUTPUT_PATH).with_name("freeform_gen_checkpoint.jsonl"))

_skip_gpu_rest_of_cell = False
if RELOAD_RESPONSES_FROM_SNAPSHOT:
    _snap_path = Path(OUTPUT_PATH).with_name("responses_snapshot.jsonl")
    if not _snap_path.is_file():
        raise FileNotFoundError(
            f"RELOAD_RESPONSES_FROM_SNAPSHOT but file not found: {_snap_path.resolve()}\n"
            "Set RELOAD_RESPONSES_FROM_SNAPSHOT=False to regenerate, or fix OUTPUT_PATH."
        )
    _by_id: dict[str, str] = {}
    with open(_snap_path, encoding="utf-8") as fp:
        for _line in fp:
            _line = _line.strip()
            if not _line:
                continue
            _row = json.loads(_line)
            _by_id[str(_row["id"])] = _row["response"]
    for _i in range(len(work_data)):
        responses[_i] = _by_id.get(str(work_data[_i]["id"]), "")
    _filled = sum(1 for _r in responses if _r)
    if _filled == 0:
        raise ValueError(
            f"Snapshot {_snap_path} produced zero filled rows (id mismatch with work_data?)."
        )
    print(f"RELOAD_RESPONSES_FROM_SNAPSHOT: loaded {_filled} non-empty rows from {_snap_path}")
    _skip_gpu_rest_of_cell = True

print(
    f"[§6] RELOAD_RESPONSES_FROM_SNAPSHOT={RELOAD_RESPONSES_FROM_SNAPSHOT!r}, "
    f"skip_gpu_generation={_skip_gpu_rest_of_cell!r}, mcq={len(mcq_idx)}, free={len(free_idx)}"
)

# Build a LoRARequest once so MCQ and free-form passes share the same adapter.
# Resolved here (not at LLM load time) so swapping ADAPTER_PATH does not require
# rebuilding the engine — vLLM hot-loads the adapter on first generate() call.
lora_request = None
if ADAPTER_PATH and not _skip_gpu_rest_of_cell:
    from vllm.lora.request import LoRARequest

    lora_request = LoRARequest("math_adapter", 1, ADAPTER_PATH)
    print(f"Using LoRA adapter: {ADAPTER_PATH}")
_gen_kwargs = {"lora_request": lora_request} if lora_request is not None else {}

# ── MCQ pass (or reload from checkpoint) ──────────────────────────────────────
if not _skip_gpu_rest_of_cell and mcq_idx:
    if SKIP_MCQ_GENERATION:
        ck_path = Path(MCQ_CHECKPOINT_PATH)
        if not ck_path.is_file():
            raise FileNotFoundError(
                f"SKIP_MCQ_GENERATION=True but file not found: {ck_path.resolve()}\n"
                "Run once with SKIP_MCQ_GENERATION=False to generate MCQ and create this checkpoint."
            )
        loaded_mcq = _load_mcq_checkpoint(str(ck_path))
        missing_ids: list[str] = []
        for i in mcq_idx:
            qid = str(work_data[i]["id"])
            if qid not in loaded_mcq:
                missing_ids.append(qid)
            else:
                responses[i] = loaded_mcq[qid]
        if missing_ids:
            raise ValueError(
                f"MCQ checkpoint missing {len(missing_ids)} id(s) (same data & SUBSET_SIZE as first run?). "
                f"Example: {missing_ids[:3]!r}"
            )
        print(f"SKIP_MCQ_GENERATION: loaded {len(mcq_idx)} MCQ from {ck_path}")
    else:
        mcq_prompts = [prompt_for_item(work_data[i]) for i in mcq_idx]
        force_stoch = GREEDY and mcq_n > 1
        sp_mcq = make_sampling_params(n=mcq_n, force_stochastic=force_stoch)
        print(f"Generating {len(mcq_prompts)} MCQ with n={mcq_n} ...")
        out_mcq = llm.generate(mcq_prompts, sampling_params=sp_mcq, **_gen_kwargs)
        for local_i, global_i in enumerate(mcq_idx):
            if mcq_n > 1:
                responses[global_i] = pick_mcq_majority_text(out_mcq[local_i].outputs)
            else:
                responses[global_i] = out_mcq[local_i].outputs[0].text.strip()
        ck_path = Path(MCQ_CHECKPOINT_PATH)
        ck_path.parent.mkdir(parents=True, exist_ok=True)
        with open(ck_path, "w", encoding="utf-8") as wf:
            for i in mcq_idx:
                wf.write(
                    json.dumps(
                        {"id": work_data[i]["id"], "response": responses[i]},
                        ensure_ascii=False,
                    )
                    + "\n"
                )
        print(f"Wrote MCQ checkpoint ({len(mcq_idx)} rows) to {ck_path}")

# ── Free-form pass ────────────────────────────────────────────────────────────
# Free-form checkpoint + responses snapshot below: AI-added persistence helpers (not original starter).

if not _skip_gpu_rest_of_cell and free_idx:
    free_prompts = [prompt_for_item(work_data[i]) for i in free_idx]
    force_stoch = GREEDY and free_n > 1
    sp_free = make_sampling_params(n=free_n, force_stochastic=force_stoch)
    print(f"Generating {len(free_prompts)} free-form with n={free_n} ...")
    out_free = llm.generate(free_prompts, sampling_params=sp_free, **_gen_kwargs)
    for local_i, global_i in enumerate(free_idx):
        if free_n > 1:
            responses[global_i] = pick_freeform_majority_text(out_free[local_i].outputs)
        else:
            responses[global_i] = out_free[local_i].outputs[0].text.strip()

    ff_ck = Path(FREEFORM_CHECKPOINT_PATH)
    ff_ck.parent.mkdir(parents=True, exist_ok=True)
    with open(ff_ck, "w", encoding="utf-8") as wf:
        for i in free_idx:
            wf.write(
                json.dumps(
                    {"id": work_data[i]["id"], "response": responses[i]},
                    ensure_ascii=False,
                )
                + "\n"
            )
    print(f"Wrote free-form checkpoint ({len(free_idx)} rows) to {ff_ck}")

# ── Snapshot all responses so far (MCQ + free-form) to Drive ──────────────────
_snapshot_path = Path(OUTPUT_PATH).with_name("responses_snapshot.jsonl")
_snapshot_path.parent.mkdir(parents=True, exist_ok=True)
with open(_snapshot_path, "w", encoding="utf-8") as wf:
    for i, resp in enumerate(responses):
        if resp:
            wf.write(
                json.dumps(
                    {"id": work_data[i]["id"], "response": resp},
                    ensure_ascii=False,
                )
                + "\n"
            )
print(f"Snapshot saved ({sum(1 for r in responses if r)} responses) to {_snapshot_path}")

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={work_data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
def score_mcq(response: str, gold_letter: str) -> bool:

    return extract_letter(response) == gold_letter.strip().upper()





import sys

from pathlib import Path





if not SAVE_EVAL:

    results = [

        {

            "id": item.get("id"),

            "is_mcq": bool(item.get("options")),

            "gold": None,

            "response": response,

            "correct": None,

        }

        for item, response in zip(work_data, responses)

    ]

    print(

        f"No labels (private split) — skipped Judger. Staged {len(results)} rows for Section 9 export."

    )

else:

    def _find_judger_repo_root() -> Path | None:

        """Locate the directory that contains judger.py (repo root)."""

        seen: set[Path] = set()

        candidates: list[Path] = []



        def push(p: Path) -> None:

            try:

                r = p.resolve()

            except OSError:

                return

            if r not in seen:

                seen.add(r)

                candidates.append(r)



        dpr = globals().get("DRIVE_PROJECT_ROOT")

        if dpr:

            push(Path(str(dpr)))

        push(Path("/content/drive/MyDrive/151b_smallLM"))

        push(Path("/content/151b_smallLM"))



        cwd = Path.cwd()

        push(cwd)

        cur: Path = cwd

        for _ in range(8):

            if cur == cur.parent:

                break

            cur = cur.parent

            push(cur)



        for base in candidates:

            if (base / "judger.py").is_file():

                return base



        mydrive = Path("/content/drive/MyDrive")

        if mydrive.is_dir():

            from collections import deque



            dq: deque[tuple[Path, int]] = deque([(mydrive, 0)])

            visited = 0

            while dq and visited < 600:

                folder, depth = dq.popleft()

                visited += 1

                try:

                    if (folder / "judger.py").is_file():

                        return folder.resolve()

                except OSError:

                    continue

                if depth >= 5:

                    continue

                try:

                    for child in folder.iterdir():

                        if child.is_dir():

                            dq.append((child, depth + 1))

                except (OSError, PermissionError):

                    pass

        return None





    import importlib.util

    import os



    # Load Judger for free-form scoring (judger.py + utils.py live at repo root).

    _repo_root = _find_judger_repo_root()

    _on_colab = importlib.util.find_spec("google.colab") is not None

    _default_drive_project = Path("/content/drive/MyDrive/151b_smallLM")



    if _repo_root is None and _on_colab and not Path("/content/drive").is_dir():

        print("Google Drive is not mounted. Mounting now (complete the browser prompt if shown)...")

        from google.colab import drive



        drive.mount("/content/drive", force_remount=False)

        globals()["DRIVE_PROJECT_ROOT"] = str(_default_drive_project)

        _default_drive_project.mkdir(parents=True, exist_ok=True)

        for sub in ("results", "checkpoints", "models"):

            (_default_drive_project / sub).mkdir(parents=True, exist_ok=True)

        if _default_drive_project.is_dir():

            os.chdir(_default_drive_project)

        _repo_root = _find_judger_repo_root()



    if _repo_root is None and _on_colab and Path("/content/drive").is_dir():

        if _default_drive_project.is_dir():

            globals()["DRIVE_PROJECT_ROOT"] = str(_default_drive_project)

            os.chdir(_default_drive_project)

            _repo_root = _find_judger_repo_root()



    if _repo_root is None:

        _drive_ok = Path("/content/drive").is_dir()

        raise FileNotFoundError(

            "judger.py not found. "

            + (

                "On Colab: authorize Google Drive when prompted, then confirm this path exists in the Files sidebar "

                f"and contains judger.py: {_default_drive_project}. "

                if _on_colab

                else "Copy the full course repo so the active folder contains judger.py and utils.py. "

            )

            + f"cwd={Path.cwd()!r}, DRIVE_PROJECT_ROOT={globals().get('DRIVE_PROJECT_ROOT')!r}, "

            f"/content/drive exists={_drive_ok}."

        )

    sys.path.insert(0, str(_repo_root))



    from judger import Judger

    judger = Judger(strict_extract=False)



    results = []

    for item, response in tqdm(zip(work_data, responses), total=len(responses), desc="Scoring"):

        is_mcq = bool(item.get("options"))

        gold   = item["answer"]



        if is_mcq:

            correct = score_mcq(response, str(gold))

        else:

            gold_list = gold if isinstance(gold, list) else [gold]

            try:

                correct = judger.auto_judge(

                    pred=response,

                    gold=gold_list,

                    options=[[]] * len(gold_list),

                )

            except Exception:

                correct = False



        results.append({

            "id":       item.get("id"),

            "is_mcq":   is_mcq,

            "gold":     gold,

            "response": response,

            "correct":  correct,

        })



    print(f"Scoring complete. {len(results)} results.")


## 8. Summary

Print accuracy broken down by question type.

In [ ]:
if SAVE_EVAL:

    mcq_res  = [r for r in results if r["is_mcq"]]

    free_res = [r for r in results if not r["is_mcq"]]



    def acc(subset):

        return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0



    print("=" * 50)

    print("EVALUATION RESULTS")

    print("=" * 50)

    print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")

    print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")

    print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")

    print("=" * 50)

else:

    print("Section 8 skipped (no labels in private split).")



## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

`SAVE_EVAL` is set in Section 2 from `DATA_SPLIT` (private → `False`). You can override it in the code cell below only if you know what you are doing.

In [ ]:
# SAVE_EVAL is set in Section 2 from DATA_SPLIT (False when DATA_SPLIT == "private").



out_path = Path(OUTPUT_PATH)

out_path.parent.mkdir(parents=True, exist_ok=True)



with open(out_path, "w") as f:

    for r in results:

        if SAVE_EVAL:

            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],

                      "response": r["response"], "correct": r["correct"]}

        else:

            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}

        f.write(json.dumps(record) + "\n")



print(f"Saved {len(results)} records to {out_path}")


## 10. Verify Google Drive Persistence

Confirm that all outputs have been written to Google Drive and will survive session termination.

*Supplementary section: written with an AI coding assistant (not original starter).*

In [ ]:
# Supplementary verification — written with an AI coding assistant (not original starter code).
from datetime import datetime

def _verify_drive_file(label: str, path: str) -> bool:
    """Check if a file exists on disk and report its size (helper added with AI assistance)."""
    p = Path(path)
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"  [OK] {label}: {p}  ({size_kb:.1f} KB)")
        return True
    print(f"  [MISSING] {label}: {p}")
    return False

print("=" * 60)
print("GOOGLE DRIVE PERSISTENCE CHECK")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

all_ok = True
all_ok &= _verify_drive_file("Final results", OUTPUT_PATH)
all_ok &= _verify_drive_file("MCQ checkpoint", MCQ_CHECKPOINT_PATH)
all_ok &= _verify_drive_file("Free-form checkpoint", FREEFORM_CHECKPOINT_PATH)
all_ok &= _verify_drive_file("Responses snapshot", str(Path(OUTPUT_PATH).with_name("responses_snapshot.jsonl")))

if DRIVE_PROJECT_ROOT:
    drive_root = Path(DRIVE_PROJECT_ROOT)
    drive_files = sorted(drive_root.rglob("*.jsonl"))
    print(f"\nAll .jsonl files under {DRIVE_PROJECT_ROOT}:")
    for f in drive_files:
        print(f"  {f.relative_to(drive_root)}  ({f.stat().st_size / 1024:.1f} KB)")

if all_ok:
    print("\nAll outputs are safely on Google Drive. You can close the browser.")
else:
    print("\nWARNING: Some files are missing! Check paths above before closing.")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!